# Parking range screen for full A2 and trained A2 Lite

Revision `PARKING-RANGE-2-6M-2026-10-21-v0.1`. Run top to bottom on a **CPU runtime**. This reads saved predictions: no training, model inference, images, checkpoint loading, Core ML or native build. The first result is a KITTI camera-depth screen, **not ultrasonic surface-distance validation**. Historical experiment gates stay unchanged.

Targets per class and one-metre band: median distance error at most 10 cm, p95 at most 20 cm, absolute bias at most 5 cm, recall and precision at least 95%. A valid-distance coverage guard also requires at least 95%. Low sample counts are inconclusive. The separate contract explains matching and evidence floors.

## 1 Mount Drive and update the project
Use the published main branch. Keep the explicit historical source paths below; edit a path only if you relocated the same completed run. Never substitute the unrecovered pruned student or a KD endpoint. Change RUN_ID if an input, source or contract changes.

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from collections import deque
import hashlib, json, os, shlex, subprocess, sys
from google.colab import drive, files
drive.mount('/content/drive')
REVISION = 'PARKING-RANGE-2-6M-2026-10-21-v0.1'  # Latest parking-only screen
RUN_ID = 'parking_range_v01_r1'
PROJECT = Path('/content/mobile_adas3d')
URL = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch','main','--single-branch',URL,str(PROJECT)],check=True)
else:
    branch = subprocess.check_output(['git','-C',str(PROJECT),'branch','--show-current'],text=True).strip()
    if branch != 'main': raise RuntimeError(f'Expected main, found {branch}; preserve this checkout')
    for args in (['diff','--quiet'], ['diff','--cached','--quiet']):
        if subprocess.run(['git','-C',str(PROJECT),*args]).returncode: raise RuntimeError('Preserve tracked edits before updating')
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'],check=True)
SCRIPT = PROJECT/'scripts/evaluate_parking_range.py'
CONTRACT = PROJECT/'configs/parking_range_validation_v0_1.json'
SPLIT = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen/val.txt')
LABELS = Path('/content/drive/MyDrive/datasets/kitti/training/label_2')
BASE = Path('/content/drive/MyDrive/mobile_adas3d_outputs')
A2_RUN = BASE/'students/monodetr_a2_m68_onnx_cpu/m68_a2_onnx_cpu_r3/export'
LITE_RUN = BASE/'edge_candidates/a2_lite_trained_onnx/a2_lite_gt_epoch10_onnx_r1'
PREDICTIONS = {'A2': A2_RUN/'fullval/predictions/onnx_cpu', 'A2_Lite': LITE_RUN/'validation/raw'}
PROVENANCE = {'A2': A2_RUN/'A2_ONNX/manifest.json', 'A2_Lite': LITE_RUN/'A2_LITE_TRAINED_ONNX/manifest.json'}
OUTPUT = BASE/'parking_validation'/RUN_ID
CACHE = Path('/content')/f'parking_text_inputs_{RUN_ID}'
LOGS = OUTPUT/'colab_logs'
for p in (SCRIPT, CONTRACT, SPLIT):
    if not p.is_file(): raise FileNotFoundError(f'Missing {p}; use the published notebook and project main')
def run_logged(command, name):
    command = [str(v) for v in command]
    LOGS.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log_path = LOGS/f'{name}_{stamp}.log'
    print('+',shlex.join(command),'\nDurable log:',log_path,flush=True)
    tail = deque(maxlen=30)
    with log_path.open('w') as log:
        with subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
            code=process.wait()
        log.write(f'\nEXIT CODE: {code}\n')
    if code: raise RuntimeError(f'Exit {code}; log={log_path}\n'+'\n'.join(tail))
    return log_path
print('Notebook revision:', REVISION)
print('Project commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT,text=True).strip())
print('Output:',OUTPUT)
print(json.dumps(json.loads(CONTRACT.read_text())['thresholds'],indent=2))


## 2 Stage labels and count available examples
Four workers copy only small text files. Verified identical local files are reused; conflicting files are preserved. The fixed Chen split is checked before copying. This inventory can run even if a model prediction directory is missing.

In [ ]:
run_logged([sys.executable,'-m','unittest','discover','-s','tests','-p','test_parking_range_validation.py','-v'],'parking_tests')
sys.path.insert(0,str(PROJECT/'scripts'))
import evaluate_parking_range as parking
contract = parking.load_config(CONTRACT)
ids = parking.ids_from_split(SPLIT,contract)
def stage_text(source, destination, names):
    if not source.is_dir(): raise FileNotFoundError(source)
    available={entry.name for entry in os.scandir(source) if entry.is_file()}
    missing=[n for n in names if n not in available]
    if missing: raise FileNotFoundError(f'{len(missing)} missing files in {source}; first: {missing[:5]}')
    destination.mkdir(parents=True,exist_ok=True)
    def copy_one(name):
        raw=(source/name).read_bytes();target=destination/name
        parking.immutable(target,raw)
        return name,hashlib.sha256(raw).hexdigest()
    copied={}
    with ThreadPoolExecutor(max_workers=4) as pool:
        for i,(name,sha) in enumerate(pool.map(copy_one,names),1):
            copied[name]=sha
            if i%500==0 or i==len(names): print(f'{source.name}: staged/verified {i}/{len(names)}',flush=True)
    return copied
label_hashes=stage_text(LABELS,CACHE/'labels',[s+'.txt' for s in ids])
parking.immutable(CACHE/'val.txt',SPLIT.read_bytes())
run_logged([sys.executable,'-u',SCRIPT,'inventory','--labels',CACHE/'labels','--split',CACHE/'val.txt','--output',OUTPUT/'inventory'],'parking_inventory')
print((OUTPUT/'inventory/inventory.csv').read_text())


## 3 Verify the exact model records and stage saved predictions
Both sources must be complete. If a path is missing, locate that completed M68 or trained-Lite run on Drive and correct the path in section 1. Do not rerun model training or select another run automatically. Hashes identify the model manifests; the supplied prediction directories must still be from those original runs.

In [ ]:
for role in ('A2','A2_Lite'):
    if not PREDICTIONS[role].is_dir(): raise FileNotFoundError(f'{role} saved predictions missing: {PREDICTIONS[role]}. No inference is needed if you locate the completed run.')
    if not PROVENANCE[role].is_file(): raise FileNotFoundError(PROVENANCE[role])
    print(role,parking.provenance(PROVENANCE[role],role,contract))
staged={}
for role in ('A2','A2_Lite'):
    staged[role]=stage_text(PREDICTIONS[role],CACHE/role,[s+'.txt' for s in ids])
    parking.immutable(CACHE/(role+'_model.json'),PROVENANCE[role].read_bytes())
parking.immutable(CACHE/'staging_manifest.json',parking.encoded({'revision':REVISION,'labels':label_hashes,'predictions':staged,'sources':{k:str(v) for k,v in PREDICTIONS.items()}}))
print('All text inputs are local. No GPU, image staging or model inference was used.')


## 4 Compare the two models
Distance and recall bins use true camera depth. Precision bins use predicted depth and require a correct match to a true 2–6 m target. These quantities are not sensor-surface range. A small or empty cell is inconclusive. A completed screen never certifies ultrasonic accuracy.

In [ ]:
command=[sys.executable,'-u',SCRIPT,'screen','--labels',CACHE/'labels','--split',CACHE/'val.txt','--output',OUTPUT/'comparison']
for role in ('A2','A2_Lite'):
    command += ['--predictions',role+'='+str(CACHE/role),'--provenance',role+'='+str(CACHE/(role+'_model.json'))]
run_logged(command,'parking_comparison')
report=json.loads((OUTPUT/'comparison/parking_range_screen.json').read_text())
for role,result in report['models'].items():
    print('\n',role,'camera-depth screen:',result['screen_status'])
    print((OUTPUT/'comparison'/role/'cells.csv').read_text())
assert report['comparison_complete'] and report['optimizer_steps']==0
assert not report['parking_validation_qualified'] and not report['ultrasonic_sensor_qualified']


## 5 Return the result bundle
Download the comparison ZIP and return it for review. It contains the contract, source hashes, inventory, per-band metrics, matches, misses and prediction assignments, but no images or model weights. Next is independently referenced mounted-camera testing, not automatic KD or deployment approval.

In [ ]:
BUNDLE=OUTPUT/'comparison/parking_range_results.zip'
if not BUNDLE.is_file(): raise FileNotFoundError('Complete section 4 first')
print('Result:',BUNDLE)
files.download(str(BUNDLE))
